# Part 1 round 2: measured development improvement

File-backed CPU analysis of already-completed Colab runs; no new GPU inference.
Separate two comparisons: round 1 vs round 2 format stability, and dense-only vs
final retrieval within each round. Human metrics remain missing. Changes followed
inspection of recorded failures, so this is development evidence, not a holdout.


In [1]:
import sys,json,hashlib,subprocess
from pathlib import Path
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'outputs/part1/round2_colab_20261114T032206Z/summary.json').is_file())
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
print('Analysis commit:',commit)
print('Analysis Python:',sys.version.split()[0],'pandas:',pd.__version__)
print('Execution: CPU analysis of saved records; source inference: Colab Tesla T4; seed: 266')
print('Original Colab source commit: unavailable, not retroactively assigned')
paths=[ROOT/'outputs/part1/round1_colab_before_schema_fix/summary.json',ROOT/'outputs/part1/round2_colab_20261114T032206Z/summary.json']
rounds=[json.loads(p.read_text(encoding='utf-8')) for p in paths]
assert rounds[0]['questions_specification_sha256']==rounds[1]['questions_specification_sha256']
print('Frozen specification:',rounds[0]['questions_specification_sha256'])


Analysis commit: 9caac5f34d2b843c5aaf5968eccd379ac2074c40
Analysis Python: 3.11.9 pandas: 3.1.1
Execution: CPU analysis of saved records; source inference: Colab Tesla T4; seed: 266
Original Colab source commit: unavailable, not retroactively assigned
Frozen specification: e4cf051593f3665cec1e411b5b745c64e33b92e9369d25c508f71e6fa01e1763


In [2]:
from IPython.display import display,Markdown
rows=[]
for run in rounds:
    for cohort in ['practice','dense_only','final']:
        r=run['practice'] if cohort=='practice' else run['golden'][cohort]
        rows.append(dict(round=run['round'],cohort=cohort,questions=r.get('questions',r.get('denominator')),
            execution_errors=r['execution_errors'],format_fallbacks=r['format_fallbacks'],
            median_seconds_per_question=r['median_seconds_per_question'],
            recall_at_5=r.get('retrieval_recall_at_5'),retrieval_denominator=r.get('retrieval_denominator'),
            answer_correctness=r.get('answer_correctness'),citation_accuracy=r.get('citation_accuracy'),
            groundedness_mean=r.get('groundedness_mean'),human_agreement=r.get('groundedness_agree_within_one')))
table=pd.DataFrame(rows)
display(table)
changes=[]
for cohort in ['practice','dense_only','final']:
    before,after=[r for r in rows if r['cohort']==cohort]
    changes.append(dict(cohort=cohort,before_fallbacks=before['format_fallbacks'],after_fallbacks=after['format_fallbacks'],
        fewer_fallbacks=before['format_fallbacks']-after['format_fallbacks'],
        reduction_percentage_points=100*(before['format_fallbacks']-after['format_fallbacks'])/after['questions']))
display(pd.DataFrame(changes))
print('Format improvements do not establish answer correctness or a higher Kaggle score.')


Format improvements do not establish answer correctness or a higher Kaggle score.


,round,cohort,questions,execution_errors,format_fallbacks,median_seconds_per_question,recall_at_5,retrieval_denominator,answer_correctness,citation_accuracy,groundedness_mean,human_agreement
0,1,practice,40,0,4,5.729000,NaN,NaN,None,None,None,None
1,1,dense_only,25,0,4,12.840790,0.472222,24.0,None,None,None,None
2,1,final,25,0,6,0.270933,0.763889,24.0,None,None,None,None
3,2,practice,40,0,0,5.126000,NaN,NaN,None,None,None,None
4,2,dense_only,25,0,2,9.173569,0.472222,24.0,None,None,None,None
5,2,final,25,0,1,0.795780,0.763889,24.0,None,None,None,None


,cohort,before_fallbacks,after_fallbacks,fewer_fallbacks,reduction_percentage_points
0,practice,4,0,4,10.0
1,dense_only,4,2,2,8.0
2,final,6,1,5,20.0


In [3]:
report='# Part 1 round 2: notebook-computed development comparison\n\n'
report+='Analysis commit: `'+commit+'`. Source GPU runs used one Tesla T4; analysis here is CPU/file-backed. Original inference Git commit was unavailable.\n\n'
report+='| Cohort | Round 1 fallbacks | Round 2 fallbacks | Fewer fallback percentage points |\n| --- | ---: | ---: | ---: |\n'
for r in changes:report+=f"| {r['cohort']} | {r['before_fallbacks']} | {r['after_fallbacks']} | {r['reduction_percentage_points']:.1f} |\n"
report+='\nWithin-round second-run required comparison:\n\n| Metric | Dense-only baseline | Final system |\n| --- | ---: | ---: |\n'
a,b=rounds[1]['golden']['dense_only'],rounds[1]['golden']['final']
for label,key in [('Questions','denominator'),('Recall@5','retrieval_recall_at_5'),('Recall denominator','retrieval_denominator'),('Median seconds/request','median_seconds_per_question'),('Answer correctness','answer_correctness'),('Citation accuracy','citation_accuracy'),('Groundedness mean','groundedness_mean'),('Human agreement within one point','groundedness_agree_within_one')]:
    fmt=lambda v:'Pending' if v is None else f'{v:.4f}' if isinstance(v,float) else str(v)
    report+=f'| {label} | {fmt(a[key])} | {fmt(b[key])} |\n'
report+='\nThis documents a measured format-stability improvement. It does not establish better answer correctness, citation accuracy or Kaggle score. Recall is unchanged between rounds. Final-system latency increased between rounds; fewer fallbacks may allow longer answers, but causal attribution is not established. Short final-system medians include deterministic structured routes, not just VLM inference.\n'
report+='\nKnown content failure: P1V2-22 reverses the correction scope. DOC3:p6 corrects the plotting error for 2012; other values remain the same. Both members must independently rate groundedness, and content/citation judgments remain pending for both systems.\n'
report+='\nFixed questions, rubrics, model revisions and seed were retained. These are development reruns after inspecting failures; do not claim an untouched holdout. The source repository is currently public; course privacy, final Git provenance, report, human review, final tag and Kaggle selection remain separate completion items.\n'
target=ROOT/'outputs/part1/round2_colab_20261114T032206Z'
(target/'ROUND2_IMPROVEMENT.md').write_text(report,encoding='utf-8')
(target/'round_comparison.json').write_text(json.dumps(dict(rows=rows,changes=changes,analysis_commit=commit,source_inference_commit=None),indent=2),encoding='utf-8')
display(Markdown(report))


# Part 1 round 2: notebook-computed development comparison

Analysis commit: `9caac5f34d2b843c5aaf5968eccd379ac2074c40`. Source GPU runs used one Tesla T4; analysis here is CPU/file-backed. Original inference Git commit was unavailable.

| Cohort | Round 1 fallbacks | Round 2 fallbacks | Fewer fallback percentage points |
| --- | ---: | ---: | ---: |
| practice | 4 | 0 | 10.0 |
| dense_only | 4 | 2 | 8.0 |
| final | 6 | 1 | 20.0 |

Within-round second-run required comparison:

| Metric | Dense-only baseline | Final system |
| --- | ---: | ---: |
| Questions | 25 | 25 |
| Recall@5 | 0.4722 | 0.7639 |
| Recall denominator | 24 | 24 |
| Median seconds/request | 9.1736 | 0.7958 |
| Answer correctness | Pending | Pending |
| Citation accuracy | Pending | Pending |
| Groundedness mean | Pending | Pending |
| Human agreement within one point | Pending | Pending |

This documents a measured format-stability improvement. It does not establish better answer correctness, citation accuracy or Kaggle score. Recall is unchanged between rounds. Final-system latency increased between rounds; fewer fallbacks may allow longer answers, but causal attribution is not established. Short final-system medians include deterministic structured routes, not just VLM inference.

Known content failure: P1V2-22 reverses the correction scope. DOC3:p6 corrects the plotting error for 2012; other values remain the same. Both members must independently rate groundedness, and content/citation judgments remain pending for both systems.

Fixed questions, rubrics, model revisions and seed were retained. These are development reruns after inspecting failures; do not claim an untouched holdout. The source repository is currently public; course privacy, final Git provenance, report, human review, final tag and Kaggle selection remain separate completion items.
